In [46]:
# importing dependencies
import pandas as pd
import matplotlib.pyplot as plt

# Dropping Phase 3 from the panel creation as NIBRS data for 2025 is not available.

In [47]:
# reading the data
ccc_p1 = pd.read_csv("ccc_compiled_20172020.csv", encoding="latin1")
ccc_p2 = pd.read_csv("ccc_compiled_20212024.csv", encoding="latin1")

/tmp/ipykernel_1857810/3468929302.py:2: DtypeWarning: Columns (6,22,31,32,33,34,35,36,37,38,39,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  ccc_p1 = pd.read_csv("ccc_compiled_20172020.csv", encoding="latin1")
/tmp/ipykernel_1857810/3468929302.py:3: DtypeWarning: Columns (22,24,26,33,34,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  ccc_p2 = pd.read_csv("ccc_compiled_20212024.csv", encoding="latin1")


#### List of variables that we are dropping and the reason to drop them

| Variable                          | Reason for dropping from analysis dataset                                                                                                                                                         |
|-----------------------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `locality`                       |We have resolved locality and county level FIPs.                                             |
| `state`                          |We have resolved state and county level FIPs.                                                |
| `county`                         | We have resolved county and county level FIPs.                                              |
| `location_detail`                | This variable is too granular and we are analysing on a county level.                       |
| `resolved_locality`              | We have county level FIPs and we use that for merging with the NIBRS dataset.               | 
| `resolved_county`                | We have county level FIPs and we use that for merging with the NIBRS dataset.               |
| `resolved_state`                 |We have county level FIPs and we use that for merging with the NIBRS dataset.                |
| `lat`                            | We have county level FIPs and we use that for merging with the NIBRS dataset.               |
| `lon`                            | We have county level FIPs and we use that for merging with the NIBRS dataset.               |
| `type`                           | Free-text categorical variable which is non-standardized and has hudnred of unique values.  |
| `chemical_agents`                | Deprecated in later phases of CCC.                                                          |
| `actors`                         | Free-text, categorical variable with hundreds of unique values and only present in phase 1. |
| `organizations`                  | Free-text, categorical variable with hundreds of unique values and only present in phase 2. |
| `participants`                   | Free-text, categorical variable with hundreds of unique values and only present in phase 2. |
| `online`                         | Dropped after keeping only offline events.                                                  |
| `title`                          | Free-text, categorical variable with hundreds of unique values and only present in phase 2. |
| `source_1`–`source_n`            | URLs to sources, mostly empty.                                                              |
| `size_low`                       | Only kept size_cat as the rest have substantial null values.                                |
| `size_high`                      | Only kept size_cat as the rest have substantial null values.                                |
| `size_mean`                      | Only kept size_cat as the rest have substantial null values.                                |
| `size_text`                      | Only kept size_cat as the rest have substantial null values.                                |
| `notes`                          | Free-text, categorical variable with hundreds of unique values.                             |
| `claims`                         | Free-text, categorical variable with hundreds of unique values and a lot of missing values. |
| `claims_summary`                 | Free-text, categorical variable with hundreds of unique values.                             |
| `claims_verbatim`                | Free-text, categorical variable with hundreds of unique values.                             |
| `issues`                         | Free-text, categorical variable with hundreds of unique values and a lot of missing values. |
| `issue_tags`                     | Free-text, categorical variable with hundreds of unique values and a lot of missing values. |
| `issue_tags_summary`             | Free-text, categorical variable with hundreds of unique values and a lot of missing values. |
| `issue_tags_verbatim`            | Free-text, categorical variable with hundreds of unique values and a lot of missing values. |
| `arrests`                        | Kept only the arrests_any variable.                                                         |
| `injuries_crowd`                 | Kept only the injuries_crowd_any variable.                                                  |
| `injuries_police`                | Kept only the injuries_police_any variable.                                                 |
| `property_damage`                | Kept only the property_damage_any variable.                                                 |

### Additional variables available but not used in the main analysis

These variables are available in the raw CCC event-level data but are **not** used in the current county–day protest–crime analysis. Some are only present (and empty) in Phase 2; others are conceptually useful but not required for our baseline estimand. We keep them in the raw data for possible future work.

| Variable              | Description                                                                                       | Why not used in main analysis (for now)                                                                                      |
|-----------------------|---------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------|
| `participant_measures`| Narrative text describing what participants did (e.g., “blocked highway”, “marched downtown”).   | Not present in Phase 1 and entirely empty in Phase 2 in our data; we therefore lack a usable, consistent measure.            |
| `police_measures`     | Narrative text describing what police did (e.g., “used tear gas”, “made arrests”).               | Not present in Phase 1 and entirely empty in Phase 2 in our data; we therefore lack a usable, consistent measure.            |
| `participant_deaths` | Text/number describing deaths among participants.                                                | Not present in Phase 1 and entirely empty in Phase 2 in our data; too sparse to use as a systematic variable at this stage.  |
| `police_deaths`      | Text/number describing deaths among police officers.                                             | Not present in Phase 1 and entirely empty in Phase 2 in our data; too sparse to use as a systematic variable at this stage.  |
| `valence`            | Numerical coding of the event’s political orientation (e.g., left/ right/ other).                | Available in both phases, but not required for the core protest–crime estimand; reserved for potential subgroup/heterogeneity analysis. |
| `macroevent`         | Identifier that links related events into a larger “macro” event.                                | Useful for studying within-macroevent dynamics, but the main analysis is at the county–day level and does not use this grouping.       |


# 1. We keep the following list of columns for analysis

In [48]:
keep_cols = [
    # --- core IDs / time ---
    "date",
    # "locality",
    # "state",
    # "location_detail",
    "online",

    # --- protest type / grouping ---
    # "type",      
    # "macroevent",
    # "actors",
    # "organizations",
    # "participants",
    # "title",

    # --- claims / issues (all dropped) ---
    # "claims",
    # "claims_summary",
    # "claims_verbatim",
    # "issues",
    # "issue_tags",
    # "issue_tags_summary",
    # "issue_tags_verbatim",

    # --- orientation ---
    # "valence",

    # --- size ---
    # "size_text",
    # "size_low",
    # "size_high",
    # "size_mean",
    "size_cat",

    # --- arrests ---
    # "arrests",
    # "arrests_any",

    # --- injuries / damage (text vs flags) ---
    # "injuries_crowd",
    "injuries_crowd_any",
    # "injuries_police",
    "injuries_police_any",
    # "property_damage",
    "property_damage_any",
    # "chemical_agents",

    # --- source URLs (all dropped) ---
    # "source_1",
    # "source_2",
    # "source_3",
    # "source_4",
    # "source_5",
    # "source_6",
    # "source_7",
    # "source_8",
    # "source_9",
    # "source_10",
    # "source_11",
    # "source_12",
    # "source_13",
    # "source_14",
    # "source_15",
    # "source_16",
    # "source_17",
    # "source_18",
    # "source_19",
    # "source_20",
    # "source_21",
    # "source_22",
    # "source_23",
    # "source_24",
    # "source_25",
    # "source_26",
    # "source_27",
    # "source_28",
    # "source_29",
    # "source_30",

    # --- notes ---
    # "notes",

    # --- geography ---
    # "lat",
    # "lon",
    # "resolved_locality",
    "resolved_county",
    # "resolved_state",
    "fips_code",

    # --- additional Phase 2-only variables (not used in main analysis) ---
    # "participant_measures",
    # "police_measures",
    # "participant_deaths",
    # "police_deaths",
]


In [49]:
# helper function to keep the important columns
def keep_columns(df, keep_cols):
    cols = [c for c in keep_cols if c in df.columns]
    return df[cols].copy()

In [50]:
ccc_p1 = keep_columns(ccc_p1, keep_cols)
ccc_p2 = keep_columns(ccc_p2, keep_cols)

# 2. Next we only keep offline events

In [51]:
ccc_p1 = ccc_p1[ccc_p1["online"] == 0].copy()
ccc_p2 = ccc_p2[ccc_p2["online"] == 0].copy()

In [52]:
# subsequently we don't require the online variable anymore
ccc_p1.drop(columns=["online"], inplace=True)
ccc_p2.drop(columns=["online"], inplace=True)

In [53]:
ccc_p1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72142 entries, 0 to 72180
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   date                 72142 non-null  object 
 1   size_cat             72142 non-null  int64  
 2   injuries_crowd_any   72142 non-null  int64  
 3   injuries_police_any  72142 non-null  int64  
 4   property_damage_any  72142 non-null  int64  
 5   resolved_county      67652 non-null  object 
 6   fips_code            72055 non-null  float64
dtypes: float64(1), int64(4), object(2)
memory usage: 4.4+ MB


In [54]:
ccc_p1.shape

(72142, 7)

In [55]:
ccc_p2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 139366 entries, 0 to 139822
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   date                 139366 non-null  object 
 1   size_cat             139366 non-null  int64  
 2   injuries_crowd_any   139366 non-null  int64  
 3   injuries_police_any  139366 non-null  int64  
 4   property_damage_any  139366 non-null  int64  
 5   resolved_county      125936 non-null  object 
 6   fips_code            139295 non-null  float64
dtypes: float64(1), int64(4), object(2)
memory usage: 8.5+ MB


In [56]:
ccc_p2.shape

(139366, 7)

# How to deal with the empty FIPS rows?

In [57]:
def check_fips_vs_county(df, name="df"):
    # rows where fips_code is missing
    mask_null_fips = df["fips_code"].isna()
    n_null_fips = mask_null_fips.sum()
    
    # among those, rows where resolved_county is also missing
    mask_both_missing = mask_null_fips & df["resolved_county"].isna()
    n_both_missing = mask_both_missing.sum()
    
    # rows where fips is missing but resolved_county is present
    n_fips_missing_county_present = n_null_fips - n_both_missing
    
    print(f"=== {name} ===")
    print("Total rows:", len(df))
    print("Rows with fips_code missing:", n_null_fips)
    print("Rows with BOTH fips_code and resolved_county missing:", n_both_missing)
    print("Rows with fips_code missing BUT resolved_county present:", n_fips_missing_county_present)
    print("All null-FIPS rows also have null resolved_county?",
          n_null_fips == n_both_missing)
    print()

# Run for both phases
check_fips_vs_county(ccc_p1, name="Phase 1 (ccc_p1)")
check_fips_vs_county(ccc_p2, name="Phase 2 (ccc_p2)")


=== Phase 1 (ccc_p1) ===
Total rows: 72142
Rows with fips_code missing: 87
Rows with BOTH fips_code and resolved_county missing: 79
Rows with fips_code missing BUT resolved_county present: 8
All null-FIPS rows also have null resolved_county? False

=== Phase 2 (ccc_p2) ===
Total rows: 139366
Rows with fips_code missing: 71
Rows with BOTH fips_code and resolved_county missing: 68
Rows with fips_code missing BUT resolved_county present: 3
All null-FIPS rows also have null resolved_county? False



### We first drop the rows where both the FIPs and the resolved county are missing as nothing can be done here.

In [58]:
ccc_p1 = ccc_p1[~(ccc_p1["fips_code"].isna() & ccc_p1["resolved_county"].isna())].copy()
ccc_p2 = ccc_p2[~(ccc_p2["fips_code"].isna() & ccc_p2["resolved_county"].isna())].copy()

In [59]:
p1_fixable = ccc_p1[ccc_p1["fips_code"].isna() & ccc_p1["resolved_county"].notna()]
p1_fixable

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
9564,2017-11-19,0,0,0,0,Norfolk,NaN
30083,2018-10-23,0,0,0,0,Metro Vancouver,NaN
42729,2019-12-06,1,0,0,0,Metro Vancouver,NaN
44816,2020-01-18,2,0,0,0,Metro Vancouver,NaN
46209,2020-03-23,1,0,0,0,LaSalle Parish,NaN
46225,2020-03-25,1,0,0,0,LaSalle Parish,NaN
52279,2020-06-06,0,0,0,0,Niasvizh District,NaN
60022,2020-08-09,2,0,0,0,Alpes-Maritimes,NaN


In [60]:
p2_fixable = ccc_p2[ccc_p2["fips_code"].isna() & ccc_p2["resolved_county"].notna()]
p2_fixable

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
14002,2021-07-15,1,0,0,0,City of Wanneroo,NaN
28354,2021-12-13,0,0,0,0,LaSalle Parish,NaN
48390,2022-06-25,2,0,0,0,Oxford County,NaN


### Internet search revealed that Metro Vancouver is in Canada, Niavizh District is in Belarus, Alpes-Maritimes is in France, City of Wanneroo is in the US.

In [61]:
bad_counties = ["Metro Vancouver", "Niasvizh District", "Alpes-Maritimes", "City of Wanneroo"]

# Phase 1
ccc_p1 = ccc_p1[~(
    ccc_p1["fips_code"].isna() &
    ccc_p1["resolved_county"].isin(bad_counties)
)].copy()

# Phase 2
ccc_p2 = ccc_p2[~(
    ccc_p2["fips_code"].isna() &
    ccc_p2["resolved_county"].isin(bad_counties)
)].copy()

In [62]:
p1_fixable = ccc_p1[ccc_p1["fips_code"].isna() & ccc_p1["resolved_county"].notna()]
p1_fixable

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
9564,2017-11-19,0,0,0,0,Norfolk,NaN
46209,2020-03-23,1,0,0,0,LaSalle Parish,NaN
46225,2020-03-25,1,0,0,0,LaSalle Parish,NaN


In [63]:
p2_fixable = ccc_p2[ccc_p2["fips_code"].isna() & ccc_p2["resolved_county"].notna()]
p2_fixable

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
28354,2021-12-13,0,0,0,0,LaSalle Parish,NaN
48390,2022-06-25,2,0,0,0,Oxford County,NaN


### We now first look at other rows with the values of LaSalle Parish, Oxford County, and Norfolk

In [64]:
ccc_p1[ccc_p1["resolved_county"] == "Oxford County"]

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
444,2017-01-21,1,0,0,0,Oxford County,23017.0
15869,2018-03-14,1,0,0,0,Oxford County,23017.0
15870,2018-03-14,1,0,0,0,Oxford County,23017.0
15881,2018-03-14,2,0,0,0,Oxford County,23017.0
31209,2018-11-08,0,0,0,0,Oxford County,23017.0
32871,2019-01-19,0,0,0,0,Oxford County,23017.0
40110,2019-09-20,2,0,0,0,Oxford County,23017.0
47131,2020-05-01,2,0,0,0,Oxford County,23017.0
53442,2020-06-09,2,0,0,0,Oxford County,23017.0
61243,2020-08-22,0,0,0,0,Oxford County,23017.0


In [65]:
ccc_p2[ccc_p2["resolved_county"] == "Oxford County"]

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
2655,2021-02-13,2,0,0,0,Oxford County,23017.0
32575,2022-01-29,0,0,0,0,Oxford County,23017.0
48042,2022-06-24,0,0,0,0,Oxford County,23017.0
48390,2022-06-25,2,0,0,0,Oxford County,NaN
50499,2022-07-13,0,0,0,0,Oxford County,23017.0
74310,2023-03-11,1,0,0,0,Oxford County,23017.0


### Assigning the row 48390 the FIPS code of 23017.0

In [66]:
ccc_p2.loc[
    (ccc_p2["resolved_county"] == "Oxford County") & ccc_p2["fips_code"].isna(),
    "fips_code"
] = 23017.0

In [67]:
ccc_p2[ccc_p2["resolved_county"] == "Oxford County"]

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
2655,2021-02-13,2,0,0,0,Oxford County,23017.0
32575,2022-01-29,0,0,0,0,Oxford County,23017.0
48042,2022-06-24,0,0,0,0,Oxford County,23017.0
48390,2022-06-25,2,0,0,0,Oxford County,23017.0
50499,2022-07-13,0,0,0,0,Oxford County,23017.0
74310,2023-03-11,1,0,0,0,Oxford County,23017.0


In [68]:
p1_fixable = ccc_p1[ccc_p1["fips_code"].isna() & ccc_p1["resolved_county"].notna()]
p1_fixable

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
9564,2017-11-19,0,0,0,0,Norfolk,NaN
46209,2020-03-23,1,0,0,0,LaSalle Parish,NaN
46225,2020-03-25,1,0,0,0,LaSalle Parish,NaN


In [69]:
p2_fixable = ccc_p2[ccc_p2["fips_code"].isna() & ccc_p2["resolved_county"].notna()]
p2_fixable

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
28354,2021-12-13,0,0,0,0,LaSalle Parish,NaN


### Now we deal with the Norfolk row

In [70]:
ccc_p1[ccc_p1["resolved_county"] == "Norfolk County"]

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
7060,2017-08-17,0,0,0,0,Norfolk County,25021.0
7210,2017-08-20,1,0,0,0,Norfolk County,25021.0
8326,2017-10-01,1,0,0,0,Norfolk County,25021.0
8973,2017-10-28,0,0,0,0,Norfolk County,25021.0
9730,2017-11-26,1,0,0,0,Norfolk County,25021.0
...,...,...,...,...,...,...,...
70576,2020-11-27,0,0,0,0,Norfolk County,25021.0
70895,2020-12-04,0,0,0,0,Norfolk County,25021.0
71273,2020-12-11,0,0,0,0,Norfolk County,25021.0
71687,2020-12-18,0,0,0,0,Norfolk County,25021.0


### Fixing the problematic row

In [71]:
ccc_p1.loc[(ccc_p1["resolved_county"] == "Norfolk") & ccc_p1["fips_code"].isna(), ["resolved_county", "fips_code"]] = ["Norfolk County", 25021.0]

### Checking again

In [72]:
p1_fixable = ccc_p1[ccc_p1["fips_code"].isna() & ccc_p1["resolved_county"].notna()]
p1_fixable

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
46209,2020-03-23,1,0,0,0,LaSalle Parish,NaN
46225,2020-03-25,1,0,0,0,LaSalle Parish,NaN


In [73]:
p2_fixable = ccc_p2[ccc_p2["fips_code"].isna() & ccc_p2["resolved_county"].notna()]
p2_fixable

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
28354,2021-12-13,0,0,0,0,LaSalle Parish,NaN


### lastly we fix the La Salle Parish rows

In [74]:
ccc_p1[ccc_p1["fips_code"] == 17099.0] # fips code for La Salle County present in Texas

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
21447,2018-04-20,0,0,0,0,LaSalle County,17099.0
26233,2018-06-30,2,0,0,0,LaSalle County,17099.0
31083,2018-11-08,0,0,0,0,LaSalle County,17099.0
37277,2019-06-22,0,0,0,0,LaSalle County,17099.0
43304,2019-12-17,0,0,0,0,LaSalle County,17099.0
46704,2020-04-20,0,0,0,0,LaSalle County,17099.0
48306,2020-05-30,1,0,0,0,LaSalle County,17099.0
49528,2020-06-01,0,0,0,0,LaSalle County,17099.0
53192,2020-06-08,2,0,0,0,LaSalle County,17099.0
53430,2020-06-09,2,0,0,0,LaSalle County,17099.0


In [75]:
ccc_p1[ccc_p1["resolved_county"] == "LaSalle Parish"]

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code
46209,2020-03-23,1,0,0,0,LaSalle Parish,NaN
46225,2020-03-25,1,0,0,0,LaSalle Parish,NaN


### La Salle Parish and La Salle County are different locations. 

In [76]:
# Phase 1
ccc_p1.loc[
    (ccc_p1["resolved_county"] == "LaSalle Parish") & ccc_p1["fips_code"].isna(),
    "fips_code"
] = 22059.0

# Phase 2
ccc_p2.loc[
    (ccc_p2["resolved_county"] == "LaSalle Parish") & ccc_p2["fips_code"].isna(),
    "fips_code"
] = 22059.0


In [77]:
p1_fixable = ccc_p1[ccc_p1["fips_code"].isna() & ccc_p1["resolved_county"].notna()]
p1_fixable

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code


In [78]:
p2_fixable = ccc_p2[ccc_p2["fips_code"].isna() & ccc_p2["resolved_county"].notna()]
p2_fixable

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,resolved_county,fips_code


# We are done with fixing the FIPS code problem.

In [79]:
ccc_p1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72058 entries, 0 to 72180
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   date                 72058 non-null  object 
 1   size_cat             72058 non-null  int64  
 2   injuries_crowd_any   72058 non-null  int64  
 3   injuries_police_any  72058 non-null  int64  
 4   property_damage_any  72058 non-null  int64  
 5   resolved_county      67647 non-null  object 
 6   fips_code            72058 non-null  float64
dtypes: float64(1), int64(4), object(2)
memory usage: 4.4+ MB


In [80]:
ccc_p2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 139297 entries, 0 to 139822
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   date                 139297 non-null  object 
 1   size_cat             139297 non-null  int64  
 2   injuries_crowd_any   139297 non-null  int64  
 3   injuries_police_any  139297 non-null  int64  
 4   property_damage_any  139297 non-null  int64  
 5   resolved_county      125935 non-null  object 
 6   fips_code            139297 non-null  float64
dtypes: float64(1), int64(4), object(2)
memory usage: 8.5+ MB


In [81]:
ccc_p1 = ccc_p1.drop(columns=["resolved_county"])
ccc_p2 = ccc_p2.drop(columns=["resolved_county"])

In [82]:
ccc_p1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72058 entries, 0 to 72180
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   date                 72058 non-null  object 
 1   size_cat             72058 non-null  int64  
 2   injuries_crowd_any   72058 non-null  int64  
 3   injuries_police_any  72058 non-null  int64  
 4   property_damage_any  72058 non-null  int64  
 5   fips_code            72058 non-null  float64
dtypes: float64(1), int64(4), object(1)
memory usage: 3.8+ MB


In [83]:
ccc_p2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 139297 entries, 0 to 139822
Data columns (total 6 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   date                 139297 non-null  object 
 1   size_cat             139297 non-null  int64  
 2   injuries_crowd_any   139297 non-null  int64  
 3   injuries_police_any  139297 non-null  int64  
 4   property_damage_any  139297 non-null  int64  
 5   fips_code            139297 non-null  float64
dtypes: float64(1), int64(4), object(1)
memory usage: 7.4+ MB


### Merging both the datasets

In [84]:
ccc_all = pd.concat([ccc_p1, ccc_p2], ignore_index=True)

In [85]:
ccc_all

,date,size_cat,injuries_crowd_any,injuries_police_any,property_damage_any,fips_code
0,2017-01-01,0,0,0,0,11001.0
1,2017-01-01,0,0,0,0,27013.0
2,2017-01-01,1,0,0,0,27053.0
3,2017-01-01,0,0,0,0,44005.0
4,2017-01-01,0,0,0,0,47001.0
...,...,...,...,...,...,...
211350,2024-12-31,0,0,0,0,51059.0
211351,2024-12-31,0,0,0,0,50023.0
211352,2024-12-31,1,0,0,0,53033.0
211353,2024-12-31,0,0,0,0,53053.0


In [86]:
# Create violent_event: 1 if any of the three is 1, else 0
ccc_all['violent_event'] = (
    (ccc_all['injuries_crowd_any'] == 1) |
    (ccc_all['injuries_police_any'] == 1) |
    (ccc_all['property_damage_any'] == 1)
).astype(int)

In [87]:
ccc_all.drop(["injuries_crowd_any", "injuries_police_any", "property_damage_any", "size_cat"], axis = 1, inplace = True)

In [88]:
ccc_all

,date,fips_code,violent_event
0,2017-01-01,11001.0,0
1,2017-01-01,27013.0,0
2,2017-01-01,27053.0,0
3,2017-01-01,44005.0,0
4,2017-01-01,47001.0,0
...,...,...,...
211350,2024-12-31,51059.0,0
211351,2024-12-31,50023.0,0
211352,2024-12-31,53033.0,0
211353,2024-12-31,53053.0,0


In [89]:
ccc_all.to_csv("ccc_panel.csv")